# Team information and work plan

| STT | Họ và tên | MSSV |
|---|---|---|
| 1 | Nguyễn Ngọc Cảnh | 23127161 |
| 2 | Lê Minh Đức | 23127351 |
| 3 | Thái Minh Khoa | 23127394 |

Nhóm phân chia sáu stage của pipeline thành ba phần. Mỗi thành viên phụ trách nghiên cứu phương pháp, triển khai, kiểm tra tính đúng và trình bày kết quả cho các stage được giao.

V0 là phiên bản C/C++ đơn luồng làm mốc so sánh. V1 chuyển toàn bộ sáu stage lên GPU bằng CUDA. V2 và V3 lần lượt bổ sung các tối ưu dựa trên kết quả profiling.

| Thành viên | Stage phụ trách | V0: Sequential | V1: Basic CUDA | V2 và V3: Optimized CUDA |
|---|---|---|---|---|
| Thái Minh Khoa | Stage 1: Preprocessing; Stage 2: Keypoint detection | Nghiên cứu, lựa chọn phương pháp và triển khai tuần tự hai stage đầu | Xây dựng CUDA kernels cho preprocessing và keypoint detection | Phân tích thời gian, đề xuất và triển khai tối ưu thuộc stage 1–2 khi được lựa chọn |
| Nguyễn Ngọc Cảnh | Stage 3: Description; Stage 4: Matching | Nghiên cứu, lựa chọn phương pháp và triển khai tuần tự descriptor và matching | Xây dựng CUDA kernels cho description và matching | Phân tích thời gian, đề xuất và triển khai tối ưu thuộc stage 3–4 khi được lựa chọn |
| Lê Minh Đức | Stage 5: Homography estimation; Stage 6: Warping and blending | Nghiên cứu, lựa chọn phương pháp và triển khai tuần tự hai stage cuối | Xây dựng CUDA kernels cho homography estimation, warping và blending | Phân tích thời gian, đề xuất và triển khai tối ưu thuộc stage 5–6 khi được lựa chọn |

**Kế hoạch dự kiến cho V2 và V3:** mỗi phiên bản bổ sung một ý tưởng tối ưu chính để đo riêng tác động. Thành viên phụ trách stage được chọn sẽ chủ trì triển khai; hai thành viên còn lại hỗ trợ kiểm tra và tích hợp. V3 kế thừa các tối ưu có hiệu quả của V2. Các phiên bản giữ cùng phương pháp và tham số để bảo đảm so sánh công bằng.

### Work plan
| Giai đoạn | Công việc và kết quả dự kiến | Người phụ trách | Mốc dự kiến |
|---|---|---|---|
| Chuẩn bị dữ liệu | Chụp ít nhất bốn cặp ảnh, gồm ba cặp điều kiện thông thường và một cặp khó; chuẩn hóa tỷ lệ 16:9 ở ba mức HD, Full HD và 4K | Khoa | 01/11 |
| Chuẩn bị notebook | Tạo bố cục báo cáo với đầy đủ các mục cần trình bày | Cảnh | 01/11 |
| Thiết lập repository | Tạo GitHub repository, tổ chức thư mục `src/` và cấp quyền cộng tác cho các thành viên | Đức | 01/11 |
| Nghiên cứu phương pháp | So sánh ưu, nhược điểm; lựa chọn phương pháp và tham số cho từng stage theo phân công | Cả nhóm | 04/11 |
| Xây dựng V0 | Triển khai các module, tích hợp đủ sáu stage và chạy pipeline tuần tự từ notebook | Cả nhóm | 07/11 |
| Hoàn thiện báo cáo giữa kỳ | Trình bày phương pháp, dữ liệu, kiểm tra tính đúng, panorama, thời gian từng stage, phân tích bottleneck và kế hoạch GPU | Cả nhóm | 07/11 |
| Xây dựng V1 | Chuyển đủ sáu stage lên GPU; đối chiếu kết quả với V0 và đo hiệu năng | Cả nhóm, theo stage phụ trách | Sau giữa kỳ |
| Xây dựng V2 | Chọn bottleneck từ profiling của V1, áp dụng một tối ưu chính và đánh giá tác động | Chủ trì bởi thành viên phụ trách stage được chọn; cả nhóm hỗ trợ | Sau V1 |
| Xây dựng V3 | Phân tích V2, bổ sung một tối ưu chính tiếp theo và đánh giá chất lượng, thời gian, speedup | Chủ trì bởi thành viên phụ trách stage được chọn; cả nhóm hỗ trợ | Sau V2 |
| Hoàn thiện bài nộp cuối kỳ | Tổng hợp so sánh các phiên bản; hoàn thiện notebook, mã nguồn, README, slides và video; kiểm tra khả năng chạy lại và đóng gói | Cả nhóm | Trước hạn nộp chính thức |

# Problem understanding

## Mô tả bài toán

Panorama image stitching là bài toán ghép các ảnh có vùng nhìn chồng lấp thành một ảnh toàn cảnh. Hai ảnh chụp cùng một cảnh từ các hướng khác nhau thường có sự khác biệt về vị trí, góc nhìn và độ sáng. Vì vậy, chương trình cần xác định những vùng tương ứng, căn chỉnh hình học và hòa trộn chúng để tạo ra ảnh đầu ra có nội dung liên tục.

Trong phạm vi đồ án, đầu vào là hai ảnh RGB có vùng chồng lấp, ký hiệu là ảnh A và ảnh B. Ảnh A được chọn làm hệ tọa độ tham chiếu. Chương trình ước lượng phép biến đổi đưa ảnh B về hệ tọa độ của ảnh A, sau đó ghép nội dung của hai ảnh trên một khung ảnh chung.

Đầu ra là ảnh panorama. Kết quả cần bảo đảm các chi tiết trong vùng chồng lấp được căn chỉnh hợp lý, đồng thời hạn chế hiện tượng ảnh đôi và đường nối rõ giữa hai ảnh.

## Các bước của pipeline

Pipeline gồm sáu bước. Mỗi bước tạo ra dữ liệu cần thiết cho bước tiếp theo:

| Bước | Vai trò trong pipeline | Kết quả đầu ra |
|---|---|---|
| **1. Preprocessing – Tiền xử lý** | Chuyển ảnh sang mức xám và làm mượt để giảm ảnh hưởng của nhiễu trước khi tìm đặc trưng. | Dữ liệu ảnh đã được tiền xử lý cho từng ảnh đầu vào. |
| **2. Keypoint detection – Phát hiện điểm đặc trưng** | Tìm các vị trí có cấu trúc nổi bật, có khả năng nhận diện lại trong ảnh còn lại. | Tập điểm đặc trưng của mỗi ảnh. |
| **3. Description – Mô tả điểm đặc trưng** | Biểu diễn vùng lân cận của mỗi điểm bằng một descriptor để có thể so sánh giữa hai ảnh. | Descriptor tương ứng với từng điểm đặc trưng. |
| **4. Matching – Ghép cặp đặc trưng** | So sánh descriptor để tìm các điểm có khả năng biểu diễn cùng một vị trí trong cảnh, đồng thời loại các ghép cặp không đủ tin cậy. | Danh sách các cặp điểm tương ứng giữa A và B. |
| **5. Homography estimation – Ước lượng phép biến đổi** | Từ các cặp điểm tương ứng, tìm phép biến đổi hình học đưa B sang A, có khả năng chịu được một số ghép cặp sai. | Ma trận homography và tập ghép cặp phù hợp với mô hình. |
| **6. Warping and blending – Biến đổi và hòa trộn ảnh** | Xác định khung ảnh chung, ánh xạ và lấy mẫu nội dung ảnh B, sau đó hòa trộn vùng chồng lấp với ảnh A. | Ảnh panorama cuối cùng. |

Ma trận homography $H_{B \rightarrow A}$ có kích thước $3 \times 3$, mô tả quan hệ giữa các điểm tương ứng trên hai ảnh trong tọa độ thuần nhất:

$$
\lambda
\begin{bmatrix}
x_A \\
y_A \\
1
\end{bmatrix}
=
H_{B \rightarrow A}
\begin{bmatrix}
x_B \\
y_B \\
1
\end{bmatrix},
$$

# Chosen methods

| Bước | Phương pháp | Tham số và lý do chọn |
|---|---|---|
| Tiền xử lý | Ảnh xám và Gaussian tách hai chiều | Ảnh xám chuẩn hóa về [0,1]; sigma 1, √2, 2, 2√2, 4. Giảm nhiễu trước khi tìm điểm. |
| Phát hiện điểm | DoG, cực trị trong vùng 3×3×3 | Ngưỡng tương phản 0.03; loại cạnh bằng Hessian; giữ tối đa 4000 điểm có đáp ứng lớn. |
| Mô tả điểm | BRIEF 256 bit | So sánh 256 cặp điểm trong vùng 31×31. Mẫu cố định với seed 21; bỏ điểm quá gần biên. |
| Ghép điểm | Khoảng cách Hamming và ratio test | Tìm hai ứng viên gần nhất; nhận khi khoảng cách nhỏ nhất < 0.8 lần khoảng cách thứ hai. |
| Homography | RANSAC và normalized DLT | 10000 lần lấy mẫu, mỗi mẫu bốn cặp điểm; ngưỡng inlier 3 pixel; tính lại H từ các inlier. |
| Biến đổi, hòa trộn | Warp ngược, nội suy song tuyến và feather blending | Tính canvas từ các góc ảnh; trộn theo khoảng cách tới biên ảnh. Pixel màu đen vẫn được xem là dữ liệu hợp lệ. |

Gaussian là phép tính theo từng pixel; BRIEF và matching có thể chia việc theo điểm đặc trưng. Các giả thuyết RANSAC độc lập với nhau, còn warp ghi một pixel đầu ra cho mỗi tác vụ. Đây là những phần có thể đưa lên GPU ở các phiên bản sau.

Các thay đổi so với ParaPano: dùng Gaussian tách hai chiều với ít mức hơn; kiểm tra cực trị đầy đủ qua không gian và mức ảnh; thêm top-K; tăng vùng BRIEF; tự viết bộ giải homography và warp thay cho OpenCV.

Bộ giải DLT dùng Jacobi trên AᵀA sau chuẩn hóa tọa độ. Cách này có hạn chế về độ ổn định số so với SVD trực tiếp. BRIEF hiện chưa bù xoay; homography cũng không xử lý được mọi sai lệch do parallax.

## 1. Tiền xử lý: ảnh xám và Gaussian separable

### Công thức

Ảnh RGB được chuyển sang ảnh xám và chuẩn hóa về khoảng $[0,1]$:

$$
I(x,y)
=
\frac{
0.299R(x,y)+0.587G(x,y)+0.114B(x,y)
}{255}.
$$

Sau đó, nhóm tạo năm mức ảnh Gaussian với:

$$
\sigma_\ell=(\sqrt{2})^\ell,
\qquad \ell=0,1,2,3,4.
$$

Các giá trị tương ứng là $1,\sqrt{2},2,2\sqrt{2},4$.

Với mỗi $\sigma_\ell$, bán kính kernel được chọn là:

$$
r_\ell=\left\lceil 3\sigma_\ell\right\rceil.
$$

Kernel Gaussian một chiều được tính và chuẩn hóa:

$$
g_\ell(t)
=
\frac{
\exp\left(-\frac{t^2}{2\sigma_\ell^2}\right)
}{
\displaystyle\sum_{j=-r_\ell}^{r_\ell}
\exp\left(-\frac{j^2}{2\sigma_\ell^2}\right)
},
\qquad -r_\ell\le t\le r_\ell.
$$

Do Gaussian hai chiều có thể tách thành tích của hai Gaussian một chiều, phép lọc được thực hiện qua hai lượt:

$$
T_\ell(x,y)
=
\sum_{t=-r_\ell}^{r_\ell}
g_\ell(t)I(x+t,y),
$$

$$
L_\ell(x,y)
=
\sum_{t=-r_\ell}^{r_\ell}
g_\ell(t)T_\ell(x,y+t).
$$

### Cách cài đặt

Hàm `preprocess_v0` thực hiện các bước:

1. Duyệt ảnh RGB và tạo mảng ảnh xám kiểu `float`.
2. Với mỗi mức, tính kernel Gaussian theo $\sigma_\ell$ và chuẩn hóa tổng trọng số về 1.
3. Lọc theo chiều ngang, lưu kết quả vào mảng tạm.
4. Lọc mảng tạm theo chiều dọc để thu được ảnh Gaussian.
5. Lưu các mức vào `PreprocessedImage.gaussian`.

Khi kernel vượt biên ảnh, tọa độ được giới hạn bằng `std::clamp`, tương đương lặp lại giá trị pixel ở biên.

Trong cài đặt hiện tại, **mỗi mức Gaussian được lọc trực tiếp từ cùng ảnh xám ban đầu**. Các mức giữ nguyên kích thước; chương trình chưa giảm độ phân giải theo octave.

### Lý do lựa chọn

Làm mượt giúp giảm ảnh hưởng của nhiễu trước khi phát hiện đặc trưng. Với kernel có chiều dài $k$, lọc Gaussian tách hai chiều giảm số phép lấy mẫu trên mỗi pixel từ khoảng $k^2$ xuống $2k$.

Mỗi pixel đầu ra trong một lượt lọc có thể được tính độc lập, phù hợp để đưa lên GPU. Hai lượt ngang và dọc vẫn cần thực hiện theo thứ tự vì lượt dọc sử dụng kết quả của lượt ngang.


## 2. Phát hiện điểm: DoG, cực trị và kiểm tra Hessian

### Difference of Gaussians

Từ năm mức Gaussian, nhóm tạo bốn ảnh sai khác:

$$
D_s(x,y)=L_{s+1}(x,y)-L_s(x,y),
\qquad s=0,1,2,3.
$$

Một điểm có phản ứng quá nhỏ được loại bằng ngưỡng tương phản:

$$
|D_s(x,y)|<0.03.
$$

Các điểm còn lại phải là cực đại hoặc cực tiểu nghiêm ngặt trong lân cận $3\times3\times3$, gồm hai chiều không gian và một chiều mức ảnh.

Cụ thể, điểm được giữ nếu giá trị của nó lớn hơn toàn bộ 26 điểm lân cận, hoặc nhỏ hơn toàn bộ 26 điểm đó. Những điểm có giá trị bằng một điểm lân cận không được xem là cực trị nghiêm ngặt.

### Loại phản ứng trên cạnh bằng Hessian

Một điểm nằm trên cạnh có thể có phản ứng mạnh nhưng khó xác định chính xác vị trí dọc theo cạnh. Nhóm dùng Hessian trên ảnh DoG để loại các điểm này:

$$
\mathcal{H}
=
\begin{bmatrix}
D_{xx} & D_{xy}\\
D_{xy} & D_{yy}
\end{bmatrix}.
$$

Các đạo hàm được xấp xỉ bằng sai phân hữu hạn tại cùng mức DoG:

$$
D_{xx}
=
D(x+1,y)+D(x-1,y)-2D(x,y),
$$

$$
D_{yy}
=
D(x,y+1)+D(x,y-1)-2D(x,y),
$$

$$
D_{xy}
=
\frac{
D(x+1,y+1)-D(x-1,y+1)
-D(x+1,y-1)+D(x-1,y-1)
}{4}.
$$

Đặt:

$$
\operatorname{tr}(\mathcal{H})=D_{xx}+D_{yy},
$$

$$
\det(\mathcal{H})=D_{xx}D_{yy}-D_{xy}^2.
$$

Điểm chỉ được giữ nếu:

$$
\det(\mathcal{H})>0
\quad\text{và}\quad
\frac{\operatorname{tr}(\mathcal{H})^2}
{\det(\mathcal{H})}
<12.1.
$$

Ngưỡng $12.1$ tương ứng với tiêu chuẩn tỷ lệ độ cong có tham số $r=10$:

$$
\frac{(r+1)^2}{r}
=
\frac{11^2}{10}
=
12.1.
$$

### Cách cài đặt

Hàm `detect_v0`:

1. Tạo bốn mảng DoG bằng cách trừ hai mức Gaussian liên tiếp.
2. Chỉ kiểm tra hai mức DoG ở giữa, $s=1$ và $s=2$, vì mỗi ứng viên cần một mức phía trước và một mức phía sau.
3. Bỏ vùng cách biên dưới 15 pixel để bảo đảm vùng mô tả BRIEF nằm trong ảnh.
4. Lần lượt kiểm tra tương phản, cực trị và Hessian.
5. Gán độ mạnh của điểm bằng:
   $$
   \operatorname{response}=|D_s(x,y)|.
   $$
6. Sắp xếp giảm dần theo `response` và giữ tối đa 4000 điểm cho mỗi ảnh.

Khi hai điểm có cùng độ mạnh, chương trình dùng tọa độ $y$, $x$ rồi mức ảnh để xác định thứ tự, giúp kết quả sắp xếp nhất quán.

Cài đặt hiện tại giữ tọa độ pixel nguyên, chưa nội suy vị trí hoặc mức ảnh của cực trị.

### Lý do lựa chọn

DoG cho phép tìm các cấu trúc nổi bật ở nhiều mức làm mượt. Ngưỡng tương phản và kiểm tra Hessian giúp loại các ứng viên yếu hoặc khó định vị. Giới hạn top-K giúp kiểm soát chi phí mô tả và matching.

Trên GPU, việc tính DoG và kiểm tra từng ứng viên có thể thực hiện song song. Thu gom các điểm hợp lệ và chọn top-K cần được xử lý riêng.


## 3. Mô tả điểm: BRIEF 256 bit

### Công thức

Với điểm đặc trưng $p=(x,y)$, BRIEF sử dụng 256 cặp độ lệch lấy mẫu $(a_i,b_i)$ trong vùng $31\times31$.

Bit thứ $i$ được xác định bằng:

$$
\tau_i(p)
=
\begin{cases}
1, & L_0(p+a_i)<L_0(p+b_i),\\
0, & \text{ngược lại}.
\end{cases}
$$

Trong đó $L_0$ là ảnh Gaussian có $\sigma=1$.

Descriptor của điểm là chuỗi nhị phân:

$$
d(p)=
\left(
\tau_0(p),\tau_1(p),\ldots,\tau_{255}(p)
\right).
$$

### Cách cài đặt

Hàm `describe_v0` sử dụng bảng `brief_pattern` cố định trong `brief_pattern.h`. Bảng được chú thích với seed 21; chương trình sử dụng trực tiếp bảng này, không tạo lại các cặp lấy mẫu ở mỗi lần chạy.

Với mỗi keypoint:

1. Kiểm tra điểm cách biên ít nhất 15 pixel.
2. Lấy ảnh làm mượt `im.gaussian[0]`.
3. So sánh cường độ của 256 cặp vị trí quanh keypoint.
4. Ghi bit 1 khi phép so sánh đúng, giữ bit 0 trong trường hợp còn lại.
5. Lưu keypoint và descriptor theo cùng thứ tự.

Mỗi descriptor được đóng gói thành tám số nguyên `uint32_t`, tương ứng 256 bit, hay 32 byte. Bit thứ $i$ nằm trong word `i / 32`, tại vị trí `i % 32`.

**Đặc điểm của cài đặt hiện tại:** tất cả keypoint đều được mô tả trên mức Gaussian có $\sigma=1$. Mức phát hiện của keypoint chưa được dùng để thay đổi kích thước vùng lấy mẫu; mẫu BRIEF cũng chưa được xoay theo hướng của điểm.

### Lý do lựa chọn

BRIEF có kích thước nhỏ và chỉ cần các phép so sánh cường độ. Descriptor nhị phân cũng cho phép matching bằng XOR và đếm bit khác nhau.

Các keypoint có thể được mô tả độc lập trên GPU. Hạn chế của cách cài đặt hiện tại là chưa bù xoay và thay đổi tỷ lệ, nên chất lượng matching có thể giảm khi hai ảnh khác nhau nhiều về các yếu tố này.


## 4. Ghép điểm: khoảng cách Hamming và ratio test

### Khoảng cách Hamming

Với hai descriptor nhị phân $d_A$ và $d_B$, khoảng cách Hamming là số vị trí bit khác nhau:

$$
d_H(d_A,d_B)
=
\sum_{i=0}^{255}
\mathbf{1}[d_A^{(i)}\ne d_B^{(i)}].
$$

Với cách đóng gói tám word 32 bit:

$$
d_H(d_A,d_B)
=
\sum_{j=0}^{7}
\operatorname{popcount}
\left(d_A[j]\oplus d_B[j]\right).
$$

Khoảng cách nằm trong đoạn $[0,256]$; giá trị nhỏ hơn thể hiện hai descriptor giống nhau hơn.

### Ratio test

Với mỗi descriptor của ảnh A, chương trình tìm hai ứng viên gần nhất trong ảnh B, có khoảng cách lần lượt là $d_1$ và $d_2$.

Ghép cặp chỉ được chấp nhận nếu:

$$
d_2>0
\quad\text{và}\quad
d_1<0.8d_2.
$$

Điều kiện này yêu cầu ứng viên tốt nhất phải khác biệt đủ rõ so với ứng viên đứng thứ hai. Nếu hai ứng viên có khoảng cách gần bằng nhau, ghép cặp được xem là không đủ tin cậy.

### Cách cài đặt

Hàm `hamming`:

1. XOR hai word tương ứng.
2. Đếm số bit 1 bằng phép cập nhật `bits &= bits - 1`.
3. Cộng kết quả của tám word.

Mỗi lần cập nhật trên xóa một bit 1 thấp nhất. V0 dùng vòng lặp tuần tự này, chưa sử dụng CUDA `__popc` hay SIMD viết tay.

Hàm `match_v0` duyệt từng descriptor của A và so sánh với toàn bộ descriptor của B. Trong khi duyệt, chương trình giữ khoảng cách tốt nhất, tốt thứ hai và chỉ số ứng viên tốt nhất.

Nếu B có ít hơn hai descriptor, hàm trả về danh sách rỗng vì không đủ dữ liệu để thực hiện ratio test.

Matching hiện tại là **một chiều từ A sang B**, chưa có cross-check. Vì vậy, nhiều điểm của A vẫn có thể được ghép với cùng một điểm của B.

### Lý do lựa chọn

Hamming phù hợp với descriptor BRIEF và có chi phí tính toán thấp. Ratio test giúp loại các trường hợp mơ hồ trước khi ước lượng homography.

Số phép so sánh descriptor tỷ lệ với $N_A N_B$, trong đó $N_A$ và $N_B$ là số điểm của hai ảnh. Đây là phần có thể chia việc theo descriptor trên GPU và khai thác phép đếm bit bằng phần cứng.


## 5. Homography: RANSAC và normalized DLT

### Mô hình biến đổi

Homography $H$ ánh xạ điểm từ ảnh B sang ảnh A:

$$
\lambda
\begin{bmatrix}
x_A\\y_A\\1
\end{bmatrix}
=
H
\begin{bmatrix}
x_B\\y_B\\1
\end{bmatrix}.
$$

Nếu:

$$
H=
\begin{bmatrix}
h_{11}&h_{12}&h_{13}\\
h_{21}&h_{22}&h_{23}\\
h_{31}&h_{32}&h_{33}
\end{bmatrix},
$$

thì tọa độ dự đoán trong ảnh A là:

$$
\hat{x}_A
=
\frac{h_{11}x_B+h_{12}y_B+h_{13}}
{h_{31}x_B+h_{32}y_B+h_{33}},
$$

$$
\hat{y}_A
=
\frac{h_{21}x_B+h_{22}y_B+h_{23}}
{h_{31}x_B+h_{32}y_B+h_{33}}.
$$

### Chuẩn hóa tọa độ

Trước khi giải DLT, nhóm chuẩn hóa riêng tập điểm của mỗi ảnh.

Với một tập gồm $n$ điểm, tính tâm:

$$
\mu_x=\frac{1}{n}\sum_i x_i,
\qquad
\mu_y=\frac{1}{n}\sum_i y_i.
$$

Khoảng cách trung bình tới tâm là:

$$
\bar{d}
=
\frac{1}{n}
\sum_i
\sqrt{(x_i-\mu_x)^2+(y_i-\mu_y)^2}.
$$

Chọn hệ số:

$$
s=\frac{\sqrt{2}}{\bar{d}},
$$

và ma trận chuẩn hóa:

$$
T=
\begin{bmatrix}
s&0&-s\mu_x\\
0&s&-s\mu_y\\
0&0&1
\end{bmatrix}.
$$

Sau chuẩn hóa, tập điểm có tâm tại gốc tọa độ và khoảng cách trung bình tới tâm bằng $\sqrt{2}$. Bước này giúp giảm chênh lệch độ lớn giữa các thành phần trong hệ phương trình.

### Giải DLT

Gọi $(x,y)$ là điểm B và $(X,Y)$ là điểm A sau chuẩn hóa. Mỗi cặp tương ứng tạo hai hàng của ma trận thiết kế $M$:

$$
\begin{bmatrix}
-x&-y&-1&0&0&0&Xx&Xy&X\\
0&0&0&-x&-y&-1&Yx&Yy&Y
\end{bmatrix}.
$$

Vector $h$ chứa chín phần tử của homography được tìm bằng:

$$
\min_{\|h\|_2=1}\|Mh\|_2^2.
$$

Cài đặt hiện tại tích lũy trực tiếp:

$$
C=M^\mathsf{T}M,
$$

rồi tìm vector riêng ứng với trị riêng nhỏ nhất của $C$ bằng phép quay Jacobi.

Mỗi bước Jacobi chọn phần tử ngoài đường chéo có trị tuyệt đối lớn nhất và dùng góc quay:

$$
\theta
=
\frac{1}{2}
\operatorname{atan2}
\left(
2C_{pq},\,C_{qq}-C_{pp}
\right).
$$

Chương trình cập nhật ma trận và các vector riêng, thực hiện tối đa 1000 phép quay hoặc dừng khi phần tử ngoài đường chéo lớn nhất nhỏ hơn $10^{-11}$.

Sau khi thu được homography chuẩn hóa $\widetilde{H}$, biến đổi về tọa độ ảnh ban đầu:

$$
H=T_A^{-1}\widetilde{H}T_B.
$$

Cuối cùng, chương trình chia toàn bộ ma trận cho $h_{33}$ để đặt phần tử này bằng 1, sau khi kiểm tra giá trị chuẩn hóa hợp lệ.

### RANSAC

Do matching vẫn có thể chứa các cặp sai, nhóm dùng RANSAC để tìm homography phù hợp với nhiều cặp điểm nhất.

Hàm `estimate_homography_v0` thực hiện:

1. Kiểm tra có ít nhất bốn ghép cặp.
2. Khởi tạo `std::mt19937` với seed 21.
3. Lặp 10000 lần; mỗi lần chọn bốn chỉ số ghép cặp khác nhau.
4. Ước lượng homography bằng normalized DLT; bỏ qua mẫu suy biến hoặc mô hình không hợp lệ.
5. Đánh giá mô hình trên toàn bộ danh sách ghép cặp.
6. Giữ mô hình có nhiều inlier nhất. Nếu số inlier bằng nhau, chọn mô hình có tổng sai số nhỏ hơn.

Sai số của ghép cặp thứ $i$ là khoảng cách chiếu lại **một chiều từ B sang A**:

$$
e_i
=
\sqrt{
(\hat{x}_{A,i}-x_{A,i})^2
+
(\hat{y}_{A,i}-y_{A,i})^2
}.
$$

Một cặp được xem là inlier nếu:

$$
e_i\le3\text{ pixel}.
$$

Sau RANSAC, chương trình fit lại homography một lần bằng toàn bộ inlier của mô hình tốt nhất, rồi đánh giá lại trên tất cả ghép cặp. Mô hình fit lại được chấp nhận nếu số inlier không giảm.

Đây là bước fit lại tuyến tính bằng DLT, chưa phải tối ưu phi tuyến theo sai số chiếu lại. Giá trị lỗi cuối cùng được báo cáo là:

$$
\bar{e}
=
\frac{1}{|\mathcal{I}|}
\sum_{i\in\mathcal{I}}e_i,
$$

với $\mathcal{I}$ là tập inlier cuối cùng. Đây là sai số Euclidean trung bình, không phải RMSE.

### Lý do lựa chọn và giới hạn

RANSAC giảm ảnh hưởng của các ghép cặp sai, còn chuẩn hóa tọa độ giúp bài toán DLT ổn định hơn. Các giả thuyết RANSAC có thể được đánh giá độc lập trên GPU.

Bộ giải hiện tại tự cài đặt Jacobi trên $M^\mathsf{T}M$. Cách này có thể kém ổn định số hơn việc dùng SVD trực tiếp trên $M$. Chương trình dùng số lần RANSAC cố định, chưa điều chỉnh số vòng lặp theo tỷ lệ inlier.


## 6. Biến đổi và hòa trộn: inverse warping, bilinear interpolation và feather blending

### Xác định canvas

Chương trình chiếu bốn góc của ảnh B qua homography $H$, sau đó lấy hình chữ nhật bao cả ảnh A và ảnh B đã biến đổi.

Các giới hạn được làm tròn ra ngoài:

$$
x_{\min}=\lfloor\min x\rfloor,
\qquad
x_{\max}=\lceil\max x\rceil,
$$

$$
y_{\min}=\lfloor\min y\rfloor,
\qquad
y_{\max}=\lceil\max y\rceil.
$$

Kích thước canvas là:

$$
W_{\text{out}}=x_{\max}-x_{\min}+1,
$$

$$
H_{\text{out}}=y_{\max}-y_{\min}+1.
$$

Pixel $(u,v)$ trên canvas tương ứng với tọa độ trong hệ ảnh A:

$$
x_A=u+x_{\min},
\qquad
y_A=v+y_{\min}.
$$

### Inverse warping

Để tìm màu đóng góp từ ảnh B, chương trình dùng:

$$
\begin{bmatrix}
\tilde{x}_B\\
\tilde{y}_B\\
\tilde{z}_B
\end{bmatrix}
=
H^{-1}
\begin{bmatrix}
x_A\\
y_A\\
1
\end{bmatrix},
$$

rồi chia cho tọa độ thuần nhất:

$$
x_B=\frac{\tilde{x}_B}{\tilde{z}_B},
\qquad
y_B=\frac{\tilde{y}_B}{\tilde{z}_B}.
$$

Như vậy, mỗi pixel đầu ra chủ động tìm vị trí lấy mẫu trong ảnh nguồn. Cách này tránh các lỗ do ánh xạ xuôi và làm tròn tọa độ.

### Nội suy song tuyến

Vị trí lấy mẫu thường không nguyên. Đặt:

$$
x_0=\lfloor x\rfloor,
\qquad
y_0=\lfloor y\rfloor,
$$

$$
\alpha=x-x_0,
\qquad
\beta=y-y_0.
$$

Trong cài đặt, hai tọa độ còn lại được giới hạn ở biên:

$$
x_1=\min(x_0+1,W-1),
\qquad
y_1=\min(y_0+1,H-1).
$$

Màu nội suy được tính riêng cho từng kênh RGB:

$$
\begin{aligned}
C(x,y)
={}&(1-\alpha)(1-\beta)C(x_0,y_0)\\
&+\alpha(1-\beta)C(x_1,y_0)\\
&+(1-\alpha)\beta C(x_0,y_1)\\
&+\alpha\beta C(x_1,y_1).
\end{aligned}
$$

Tọa độ nằm ngoài ảnh nguồn không tạo đóng góp cho pixel đầu ra.

### Feather blending

Với tọa độ hợp lệ trong một ảnh nguồn kích thước $W\times H$, trọng số được tính bằng:

$$
w(x,y)
=
1+\min(x,y,W-1-x,H-1-y).
$$

Nếu tọa độ nằm ngoài ảnh, trọng số bằng 0.

Đây là trọng số theo khoảng cách tới biên của **từng ảnh nguồn**. Pixel ở sâu trong ảnh có trọng số lớn hơn pixel gần biên; số hạng 1 giúp pixel hợp lệ ngay tại biên vẫn có trọng số dương.

Gọi $C_A,C_B$ là màu lấy mẫu và $w_A,w_B$ là trọng số tương ứng. Màu đầu ra là:

$$
C_{\text{out}}
=
\frac{w_AC_A+w_BC_B}{w_A+w_B},
\qquad w_A+w_B>0.
$$

Nếu chỉ một ảnh hợp lệ, công thức trả về màu của ảnh đó. Nếu cả hai ảnh đều không hợp lệ, pixel giữ màu đen.

### Cách cài đặt

Hàm `warp_blend_v0`:

1. Tính $H^{-1}$ và kiểm tra homography có khả nghịch.
2. Chiếu các góc để xác định canvas; từ chối trường hợp đường chân trời xạ ảnh cắt ảnh B hoặc canvas vượt giới hạn cho phép.
3. Duyệt từng pixel trên canvas.
4. Lấy màu và trọng số từ ảnh A tại $(x_A,y_A)$.
5. Dùng $H^{-1}$ để tìm $(x_B,y_B)$, rồi lấy màu và trọng số từ ảnh B.
6. Hòa trộn, làm tròn và giới hạn giá trị từng kênh về $[0,255]`.

Tính hợp lệ được xác định bằng tọa độ lấy mẫu, không dựa vào màu pixel. Vì vậy, pixel màu đen nằm trong ảnh vẫn là dữ liệu hợp lệ và tham gia hòa trộn.

### Lý do lựa chọn và giới hạn

Inverse warping cho phép xử lý độc lập từng pixel đầu ra. Nội suy song tuyến lấy màu từ các pixel lân cận thay vì chỉ chọn một pixel gần nhất. Feather blending có chi phí thấp và giúp làm mềm đường nối.

Tuy nhiên, feather blending không sửa được lỗi căn chỉnh, parallax hoặc ảnh đôi do vật thể chuyển động. Cài đặt hiện tại cũng chưa có bước bù chênh lệch phơi sáng hay hòa trộn đa băng tần.


# Sequential version (V0)

In [ ]:
!ls

In [ ]:
!unzip

In [ ]:
!unzip src.zip

### Chuẩn bị môi trường

In [ ]:
from pathlib import Path
import os

paths = [Path.cwd(), Path.cwd() / "Midterm",
         Path("/content/CSC14120-Parallel-Programming/Midterm")]
base = next(path for path in paths if (path / "src/common.h").is_file())
os.chdir(base)
data = Path(os.environ.get("PANORAMA_DATASET", "Dataset"))
if not data.is_dir():
    data = Path("Dataset")
assert data.is_dir(), "Chưa có thư mục Dataset"
os.environ["PANORAMA_DATASET"] = str(data.resolve())
Path("build").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

out = Path("outputs")

In [ ]:
!g++ --version
!nvcc --version

### Run

In [ ]:
%%bash
set -eu
g++ -O2 -std=c++17 -c src/stb_impl.cpp -o build/stb_impl.o
for file in src/v0/*.cpp; do
    name=$(basename "$file" .cpp)
    g++ -O2 -std=c++17 -c "$file" -o "build/v0_${name}.o"
done

In [ ]:
!nvcc -O2 -std=c++17 -c src/main.cu -o build/main.o

In [ ]:
!nvcc build/main.o build/stb_impl.o build/v0_*.o -o stitch

In [ ]:
!g++ -O2 -std=c++17 src/tests.cpp build/v0_*.o -o tests

# V0 results and profiling

# Dataset

# GPU plan